In [2]:
taxi_year    = 2024         
taxi_months  = "1,2,3,4,5,6,7,8,9,10,11,12" 
overwrite    = False       

StatementMeta(, 695f8732-0a89-46b5-8035-9539215a6f1e, 4, Finished, Available, Finished, False)

In [3]:
import json
from datetime import datetime, timezone

def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()

run_log = {
    "run_id":   datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
    "started":  utcnow(),
    "sources":  {},
}

def log_step(name: str, status: str, details: dict = None):
    run_log["sources"][name] = {
        "status":    status,
        "timestamp": utcnow(),
        **(details or {}),
    }
    icon = "OK" if status == "success" else "ERROR"
    print(f"  {icon}  {name:25s} → {status}")

StatementMeta(, 695f8732-0a89-46b5-8035-9539215a6f1e, 5, Finished, Available, Finished, False)

In [4]:
print(f"\n{'='*60}")
print(f"  Bronze Orchestrator – Run {run_log['run_id']}")
print(f"{'='*60}\n")

print("1/4  NYC Taxi Parquet …\n")
try:
    notebookutils.notebook.run(
        "01_bronze_taxi_ingestion",
        timeout_seconds=3600,
        arguments={
            "year":      taxi_year,
            "months":    taxi_months,   # string "1,2,3,...,12" — scalars only
            "overwrite": overwrite,
        },
    )
    log_step("nyc_taxi", "success")
except Exception as e:
    log_step("nyc_taxi", "failed", {"error": str(e)})
    print(f"  [ERROR] Taxi ingestion failed: {e}")

StatementMeta(, 695f8732-0a89-46b5-8035-9539215a6f1e, 6, Finished, Available, Finished, False)


  Bronze Orchestrator – Run 20260522T065109Z

1/4  NYC Taxi Parquet …



  OK  nyc_taxi                  → success


In [5]:
print("\n2/4  OpenAQ Air Quality …\n")
try:
    notebookutils.notebook.run(
        "02_bronze_openaq_ingestion",
        timeout_seconds=1800,
    )
    log_step("openaq", "success")
except Exception as e:
    log_step("openaq", "failed", {"error": str(e)})
    print(f"  [ERROR] OpenAQ ingestion failed: {e}")

StatementMeta(, 695f8732-0a89-46b5-8035-9539215a6f1e, 7, Finished, Available, Finished, False)


2/4  OpenAQ Air Quality …



  OK  openaq                    → success


In [6]:
print("\n3/4  World Bank GDP …\n")
try:
    notebookutils.notebook.run(
        "03_bronze_gdp_ingestion",
        timeout_seconds=600,
    )
    log_step("world_bank_gdp", "success")
except Exception as e:
    log_step("world_bank_gdp", "failed", {"error": str(e)})
    print(f"  [ERROR] GDP ingestion failed: {e}")

StatementMeta(, 695f8732-0a89-46b5-8035-9539215a6f1e, 8, Finished, Available, Finished, False)


3/4  World Bank GDP …



  OK  world_bank_gdp            → success


In [7]:
print("\n4/4  ECB FX Rates …\n")
try:
    notebookutils.notebook.run(
        "04_bronze_ecb_fx_ingestion",
        timeout_seconds=300,
    )
    log_step("ecb_fx", "success")
except Exception as e:
    log_step("ecb_fx", "failed", {"error": str(e)})
    print(f"  [ERROR] ECB FX ingestion failed: {e}")

StatementMeta(, 695f8732-0a89-46b5-8035-9539215a6f1e, 9, Finished, Available, Finished, False)


4/4  ECB FX Rates …



  OK  ecb_fx                    → success


In [8]:
run_log["finished"] = utcnow()

successes = sum(1 for v in run_log["sources"].values() if v["status"] == "success")
failures  = sum(1 for v in run_log["sources"].values() if v["status"] == "failed")

print(f"\n{'='*60}")
print(f"  BRONZE INGESTION COMPLETE")
print(f"  Passed : {successes}/4 sources")
print(f"  Failed : {failures}/4 sources")
print(f"  Run ID : {run_log['run_id']}")
print(f"{'='*60}\n")

print(json.dumps(run_log, indent=2))

# Optionally persist run log to Bronze Files/
import os
log_dir = "/lakehouse/default/Files/_run_logs"
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, f"bronze_run_{run_log['run_id']}.json")
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=2)
print(f"\n  [OK] Run log saved: {log_path}")

# Surface failures to the calling pipeline if any
if failures > 0:
    raise RuntimeError(
        f"Bronze ingestion completed with {failures} failure(s). "
        f"Check run log: {log_path}"
    )

StatementMeta(, 695f8732-0a89-46b5-8035-9539215a6f1e, 10, Finished, Available, Finished, False)


  BRONZE INGESTION COMPLETE
  Passed : 4/4 sources
  Failed : 0/4 sources
  Run ID : 20260522T065109Z

{
  "run_id": "20260522T065109Z",
  "started": "2026-05-22T06:51:09.172611+00:00",
  "sources": {
    "nyc_taxi": {
      "status": "success",
      "timestamp": "2026-05-22T06:51:22.959938+00:00"
    },
    "openaq": {
      "status": "success",
      "timestamp": "2026-05-22T06:56:12.443618+00:00"
    },
    "world_bank_gdp": {
      "status": "success",
      "timestamp": "2026-05-22T06:56:36.098765+00:00"
    },
    "ecb_fx": {
      "status": "success",
      "timestamp": "2026-05-22T06:56:57.922189+00:00"
    }
  },
  "finished": "2026-05-22T06:56:59.479059+00:00"
}

  [OK] Run log saved: /lakehouse/default/Files/_run_logs/bronze_run_20260522T065109Z.json
